In [ ]:
# Learning Objective
# 1. What data quality means
# 2. Missing-value checks
# 3. Invalid-value checks
# 4. Duplicate checks
# 5. Range validation
# 6. Timestamp validation
# 7. Combining checks
# 8. Returning results as dictionaries
# 9. Building a reusable quality_checks.py
# 10. Running it against your EnergyIQ dataset
# 11. Writing tests for the quality engine

import pandas as pd
import numpy as np

In [ ]:
# Creating a small dataset
data = {
    "timestamp": [
        "2026-01-01 10:00",
        "2026-01-01 10:01",
        "2026-01-01 10:02",
        "2026-01-01 10:03",
        "2026-01-01 10:03" 
    ],
    "power": [
        1.2,
        1.4,
        np.nan,
        -2.5,
        1.8
    ]
}

df = pd.DataFrame(data)
print(df)

          timestamp  power
0  2026-01-01 10:00    1.2
1  2026-01-01 10:01    1.4
2  2026-01-01 10:02    NaN
3  2026-01-01 10:03   -2.5
4  2026-01-01 10:03    1.8


In [ ]:
# Convert timestamp (is str now) to datatime
print(df.dtypes)

timestamp        str
power        float64
dtype: object


In [ ]:
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(df.dtypes)
# This conversion will help in time-series operations as Pandas now will understand year, month,
# day, hour, minute

timestamp    datetime64[us]
power               float64
dtype: object


In [7]:
# Check for null value
print(df.isna())

   timestamp  power
0      False  False
1      False  False
2      False   True
3      False  False
4      False  False


In [8]:
missing_power = df["power"].isna().sum()
print(missing_power)

1


In [11]:
# making it a function
def check_missing_values(df):
    missing = df["power"].isna().sum()

    return missing    

In [12]:
result = check_missing_values(df)

print(result)

1


In [17]:
# Enhancing to only report missing column
def check_missing_values(df):
    missing = df["power"].isna().sum()
    
    missing = missing[missing > 0]
    
    return missing    

print(check_missing_values(df))


[1]


In [ ]:
# Check for duplicates
#print(df.duplicated())

duplicate_count = df.duplicated().sum()

print(duplicate_count)

0


In [21]:
# check duplicates for specific columns
duplicate_timestamps = df["timestamp"].duplicated().sum()

print(duplicate_timestamps)

1


In [22]:
# Creating function

def check_duplicate_timestamps(df):
    duplicate_count = df["timestamp"].duplicated().sum()
    
    return duplicate_count
    

result = check_duplicate_timestamps(df)
print(result)

1


In [23]:
# function for negative values
def check_negative_power(df):
    negative_count = (df["power"] < 0).sum()

    return negative_count

print(check_negative_power(df))

1


In [24]:
# condition1 | condition2 and condition1 & condition2
(df["power"] < 0) | (df["power"] > 10)

df[
    (df["power"] > 2) &
    (df["power"] < 5)
]

,timestamp,power


In [25]:
# Creating range validation function

def check_power_range(df, minimum=0, maximum=10):

    invalid = df[
        (df["power"] < minimum) |
        (df["power"] > maximum)
    ]

    return len(invalid)


result = check_power_range(df)

print(result)

1


In [26]:
# Check missing timestamps
def check_missing_timestamps(df):
    return df["timestamp"].isna().sum()

In [28]:
# Timestamp ordering

is_sorted = df["timestamp"].is_monotonic_increasing
print(is_sorted)

True


In [29]:
# Delibrate break
df_test = df.iloc[::-1]

print(df_test)


            timestamp  power
4 2026-01-01 10:03:00    1.8
3 2026-01-01 10:03:00   -2.5
2 2026-01-01 10:02:00    NaN
1 2026-01-01 10:01:00    1.4
0 2026-01-01 10:00:00    1.2


In [30]:
print(df_test["timestamp"].is_monotonic_increasing)

False


In [31]:
# turning it into function
def check_timestamp_order(df):
    return df["timestamp"].is_monotonic_increasing

print(check_timestamp_order(df))

True


In [32]:
# Building quality engine

def run_quality_checks(df):

    result = {
        "missing_values": check_missing_values(df),
        "duplicate_timestamps": check_duplicate_timestamps(df),
        "negative_power": check_negative_power(df),
        "invalid_power_range": check_power_range(df),
        "missing_timestamps": check_missing_timestamps(df),
        "timestamps_sorted": check_timestamp_order(df)
    }

    return result


quality_result = run_quality_checks(df)

print(quality_result)

{'missing_values': array([1]), 'duplicate_timestamps': np.int64(1), 'negative_power': np.int64(1), 'invalid_power_range': 1, 'missing_timestamps': np.int64(0), 'timestamps_sorted': True}


In [33]:
# instead of returning Panda series, lets return number
def check_missing_values(df):
    return int(df.isna().sum().sum())


print(check_missing_values(df))

1


In [34]:
# Enhancing quality engine

def run_quality_checks(df):

    result = {
        "total_rows": len(df),
        "missing_values": check_missing_values(df),
        "duplicate_timestamps": check_duplicate_timestamps(df),
        "negative_power": check_negative_power(df),
        "invalid_power_range": check_power_range(df),
        "missing_timestamps": check_missing_timestamps(df),
        "timestamps_sorted": check_timestamp_order(df)
    }

    return result


quality_result = run_quality_checks(df)

print(quality_result)

{'total_rows': 5, 'missing_values': 1, 'duplicate_timestamps': np.int64(1), 'negative_power': np.int64(1), 'invalid_power_range': 1, 'missing_timestamps': np.int64(0), 'timestamps_sorted': True}


In [35]:
# Adding pass/fail

def run_quality_checks(df):

    missing = check_missing_values(df)
    duplicates = check_duplicate_timestamps(df)
    negative = check_negative_power(df)
    invalid_range = check_power_range(df)
    missing_timestamps = check_missing_timestamps(df)
    sorted_timestamps = check_timestamp_order(df)

    overall_status = "PASS"

    if missing > 0:
        overall_status = "FAIL"

    if duplicates > 0:
        overall_status = "FAIL"

    if negative > 0:
        overall_status = "FAIL"

    if invalid_range > 0:
        overall_status = "FAIL"

    if missing_timestamps > 0:
        overall_status = "FAIL"

    if not sorted_timestamps:
        overall_status = "FAIL"

    result = {
        "total_rows": len(df),
        "missing_values": missing,
        "duplicate_timestamps": duplicates,
        "negative_power": negative,
        "invalid_power_range": invalid_range,
        "missing_timestamps": missing_timestamps,
        "timestamps_sorted": sorted_timestamps,
        "overall_status": overall_status
    }

    return result


result = run_quality_checks(df)
print(result)



{'total_rows': 5, 'missing_values': 1, 'duplicate_timestamps': np.int64(1), 'negative_power': np.int64(1), 'invalid_power_range': 1, 'missing_timestamps': np.int64(0), 'timestamps_sorted': True, 'overall_status': 'FAIL'}


In [53]:
# Now with real data set
df = pd.read_csv(
    "../data/raw/household_power_consumption.txt",
    sep=";",
    na_values="?"
)

df["timestamp"] = pd.to_datetime(
    df["Date"] + " " + df["Time"],
    dayfirst=True
)

# #print head
# print(df.head())
# print(df.describe())
# print(df.info())
print(df)
# printing columns
#print(df.columns)

               Date      Time  Global_active_power  Global_reactive_power  \
0        16/12/2006  17:24:00                4.216                  0.418   
1        16/12/2006  17:25:00                5.360                  0.436   
2        16/12/2006  17:26:00                5.374                  0.498   
3        16/12/2006  17:27:00                5.388                  0.502   
4        16/12/2006  17:28:00                3.666                  0.528   
...             ...       ...                  ...                    ...   
2075254  26/11/2010  20:58:00                0.946                  0.000   
2075255  26/11/2010  20:59:00                0.944                  0.000   
2075256  26/11/2010  21:00:00                0.938                  0.000   
2075257  26/11/2010  21:01:00                0.934                  0.000   
2075258  26/11/2010  21:02:00                0.932                  0.000   

         Voltage  Global_intensity  Sub_metering_1  Sub_metering_2  \
0    

In [54]:
# Check for missing values
missing_values = df.isna().sum()
print(missing_values)

Date                         0
Time                         0
Global_active_power      25979
Global_reactive_power    25979
Voltage                  25979
Global_intensity         25979
Sub_metering_1           25979
Sub_metering_2           25979
Sub_metering_3           25979
timestamp                    0
dtype: int64


In [41]:
# missing value percentage

missing_count = df.isna().sum()
missing_percentage = (missing_count/len(df)) * 100

print(missing_percentage)

Date                     0.000000
Time                     0.000000
Global_active_power      1.251844
Global_reactive_power    1.251844
Voltage                  1.251844
Global_intensity         1.251844
Sub_metering_1           1.251844
Sub_metering_2           1.251844
Sub_metering_3           1.251844
timestamp                0.000000
dtype: float64


In [42]:
# Generating report
quality_report = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percentage": missing_percentage
})

print(quality_report)

                       missing_count  missing_percentage
Date                               0            0.000000
Time                               0            0.000000
Global_active_power            25979            1.251844
Global_reactive_power          25979            1.251844
Voltage                        25979            1.251844
Global_intensity               25979            1.251844
Sub_metering_1                 25979            1.251844
Sub_metering_2                 25979            1.251844
Sub_metering_3                 25979            1.251844
timestamp                          0            0.000000


In [43]:
# Check duplicates
duplicate_rows = df.duplicated().sum()
print(duplicate_rows)

0


In [46]:
# duplicate timesstamp rows
duplicate_timestamps = df["timestamp"].duplicated().sum()
print("Duplicated Timestamps", duplicate_timestamps)

Duplicated Timestamps 0


In [55]:
# Check for negative values across columns
numeric_columns = [
    "Global_active_power",
    "Global_reactive_power",
    "Global_intensity",
    "Sub_metering_1",
    "Sub_metering_2",
    "Sub_metering_3"
]

for column in numeric_columns:
    negative_count = ( df[column] < 0 ).sum()
    

print(
    column,
    negative_count
)
    

Sub_metering_3 0


In [56]:
# Making a resuable function
def check_negative_values(df, columns):

    result = {}

    for column in columns:

        negative_count = (
            df[column] < 0
        ).sum()

        result[column] = int(
            negative_count
        )

    return result


negative_results = check_negative_values(
    df,
    numeric_columns
)

print(negative_results)

{'Global_active_power': 0, 'Global_reactive_power': 0, 'Global_intensity': 0, 'Sub_metering_1': 0, 'Sub_metering_2': 0, 'Sub_metering_3': 0}


In [60]:
# Fixing issue while adding from src.quality_checks from src folder
# ---------------------------------------------------------------------------
# ModuleNotFoundError                       Traceback (most recent call last)
# Cell In[59], line 2
#       1 # Import into Notebook
# ----> 2 from src.quality_checks import run_quality_checks
#       3 
#       4 result = run_quality_checks(
#       5     df,

# ModuleNotFoundError: No module named 'src'

import os
print(os.getcwd())

c:\repo\energyiq\notebooks


In [64]:
#Since your notebook is inside notebooks, add the project root to Python's path:
# Make sure src contains __init__.py and it can be completely empty
import sys
import os

# sys.path.append(os.path.abspath(".."))
# from src.quality_checks import run_quality_checks

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.quality_checks import run_quality_checks

In [65]:

# Import into Notebook
#from src.quality_checks import run_quality_checks

result = run_quality_checks(
    df,
    numeric_columns
)

print(result)


{'total_rows': 2075259, 'missing_values': 181853, 'duplicate_timestamps': 0, 'timestamps_sorted': True, 'negative_values': {'Global_active_power': 0, 'Global_reactive_power': 0, 'Global_intensity': 0, 'Sub_metering_1': 0, 'Sub_metering_2': 0, 'Sub_metering_3': 0}}
